# Stiffness measurement processing

Merge stiffness measurements with pathologist spot annotations, keep a target annotation class (e.g. `Stroma`), remove statistical outliers per capture spot using MAD/CV filtering, and write the cleaned measurements to a TSV file. Also produces a QC histogram of the number of measurements per capture spot.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os
import numpy as np
from pathlib import Path
from matplotlib.ticker import MaxNLocator
import matplotlib as mpl
from sklearn.preprocessing import RobustScaler

new_rc_params = {'text.usetex': False, "svg.fonttype": 'none'}
mpl.rcParams.update(new_rc_params)

## Parameters

In [ ]:
# --- Input paths ---
#  File with stiffness measurements
stiffness_file = ""

# Directory containing per-section pathologist annotation CSV files
annotation_dir = ""

# --- Output paths ---
# Where to write the filtered measurements TSV file
output_tsv = ""

# Where to save the QC histogram
output_fig = ""

# --- Analysis settings ---
annotation_label = "Stroma"   # pathologist annotation label to keep
value_column = "Y"           # numeric measurement column to filter on
radius = 50                  # radius of the spot area 
min_measurements = 2         # minimum number of measurements a spot must have to be kept
mad_threshold = 3.5          # modified z-score threshold (spots with > 2 measurements)
cv_threshold = 40            # coefficient-of-variation %% threshold (spots with exactly 2 measurements)

## Load and merge data

In [ ]:
# Read stiffness measurements
df = pd.read_excel(stiffness_file, index_col=0)

In [ ]:
# Add section (sample) to the barcode
df['Barcode'] = df['section'] + '_' + df['Barcode']

In [ ]:
# Keep measurements within the spot area
df=df[df['Distance_um']<=radius]

In [ ]:
# Add spot pathologist annotation
def load_and_annotate_csv_files(folder_path):
    """Load per-section annotation CSV files and prefix barcodes with the section name.

    Each CSV file is expected to be named like '<prefix>_something.csv' and to
    contain two columns (Barcode, Annotation) with a header row.
    """
    csv_files = [f for f in os.listdir(folder_path) if f.endswith('.csv')]
    if not csv_files:
        raise FileNotFoundError(f"No CSV files found in {folder_path}")

    annotated_dfs = []
    for csv_file in csv_files:
        file_df = pd.read_csv(os.path.join(folder_path, csv_file), names=['Barcode', 'Annotation'], skiprows=1)
        prefix = csv_file.split('_')[0]
        file_df['Barcode'] = file_df['Barcode'].apply(lambda x: f"{prefix}_{x}")
        annotated_dfs.append(file_df)

    return pd.concat(annotated_dfs, ignore_index=True)

# Load and annotate the CSV files
annotated_df = load_and_annotate_csv_files(annotation_dir)

In [ ]:
# Merge with combined_df on the 'Barcode' column
combined_df = pd.merge(df, annotated_df, on='Barcode', how='left')

# Keep only the target annotation label
combined_df = combined_df[combined_df['Annotation'] == annotation_label]
combined_df = combined_df.reset_index(drop=True)

## QC: measurements per spot

In [ ]:
# Plot measurements per spot
# Count the occurrences of each unique barcode
barcode_counts = combined_df['Barcode'].value_counts()

# Calculate the median of the counts
median_count = barcode_counts.median()

# Plot the histogram
plt.figure(figsize=(10, 6))
plt.hist(barcode_counts, bins=range(1, barcode_counts.max() + 2), alpha=0.7, color='blue', edgecolor='black')
plt.axvline(median_count, color='red', linestyle='dashed', linewidth=1, label='Median')
plt.title('Counts of measurements across capture spots')
plt.xlabel('Counts of Measurements')
plt.ylabel('Frequency')
plt.legend(frameon=False)

if output_fig is not None:
    output_fig.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(output_fig)

plt.show()

In [ ]:
# Calculate the counts of each unique value in the 'Barcode' column
barcode_counts = combined_df['Barcode'].value_counts()

# Identify barcodes that appear fewer times than the minimum required
barcodes_to_keep = barcode_counts[barcode_counts >= min_measurements].index

# Keep rows where the 'Barcode' value appears the minimum number of times or more
df = combined_df[combined_df['Barcode'].isin(barcodes_to_keep)]

# Filtering of measurements based on MAD and CV

In [ ]:
# Function to compute Median Absolute Deviation (MAD) based outliers
def apply_mad(dataframe, column, threshold=mad_threshold):
    """Flag rows in `dataframe[column]` whose modified z-score exceeds `threshold`."""
    # Scale data using RobustScaler
    scaler = RobustScaler()
    reshaped_data = dataframe[column].values.reshape(-1, 1)
    scaled_data = scaler.fit_transform(reshaped_data)

    # Calculate MAD
    median = np.median(scaled_data)
    diff = np.abs(scaled_data - median)
    mad = np.median(diff)

    if mad == 0:
        # Avoid division by zero when all values are identical
        return dataframe.iloc[0:0]

    # Define outliers as those where the deviation is more than 'threshold' times the MAD
    modified_z_score = 0.6745 * diff / mad
    return dataframe[modified_z_score.flatten() > threshold]

# Function to compute Coefficient of Variation (CV) based filtering for two measurements
def apply_cv_filter(dataframe, column, cv_threshold):
    """Flag a pair of measurements as outliers if their CV exceeds `cv_threshold`."""
    mean_value = np.mean(dataframe[column])
    std_dev = np.std(dataframe[column])
    cv = std_dev * 100 / mean_value if mean_value != 0 else 0
    if cv > cv_threshold:
        # If the CV is too high, consider both points as outliers
        return dataframe  # Return both measurements
    else:
        return pd.DataFrame()  # No outliers

unique_barcodes = df['Barcode'].unique()
# Collect non-outlier data
filtered_chunks = []

for barcode in unique_barcodes:
    data = df[df['Barcode'] == barcode]
    # Remove outliers using MAD value if more than 2 measurements
    if len(data) > 2:
        outliers_df = apply_mad(data, value_column, threshold=mad_threshold)
        data = data[~data.index.isin(outliers_df.index)]
        filtered_chunks.append(data)
    elif len(data) == 2:  # If there are exactly 2 measurements, apply CV-based filter
        outliers_df = apply_cv_filter(data, value_column, cv_threshold=cv_threshold)
        data = data[~data.index.isin(outliers_df.index)]
        filtered_chunks.append(data)
    else:
        filtered_chunks.append(data)

filtered_data = pd.concat(filtered_chunks, ignore_index=True) if filtered_chunks else df.iloc[0:0]

## Save filtered measurements

In [ ]:
output_tsv.parent.mkdir(parents=True, exist_ok=True)
filtered_data.to_csv(output_tsv, sep='\t', index=False)
print(f"Wrote {len(filtered_data)} filtered measurements to {output_tsv}")